In [1]:
#2.2 编程题
import numpy as np

def max_pool2d(input, kernel_size, stride, padding):
    """
    二维最大池化前向传播
    input: (N, C, H, W) 或 (C, H, W) 的 numpy 数组
    kernel_size: int 或 (kh, kw)，池化窗口大小
    stride: int 或 (sh, sw)
    padding: int 或 (ph, pw)，零填充数量
    返回 output: numpy array，形状 (N, C, H_out, W_out)
    """
    # 统一转换为四维 (N, C, H, W)
    if input.ndim == 3:
        input = input[np.newaxis, ...]  # (1, C, H, W)
    N, C, H, W = input.shape
    
    # 处理参数
    if isinstance(kernel_size, int):
        kh = kw = kernel_size
    else:
        kh, kw = kernel_size
    if isinstance(stride, int):
        sh = sw = stride
    else:
        sh, sw = stride
    if isinstance(padding, int):
        ph = pw = padding
    else:
        ph, pw = padding
    
    # 零填充
    padded_H = H + 2 * ph
    padded_W = W + 2 * pw
    padded_input = np.pad(input, ((0,0), (0,0), (ph, ph), (pw, pw)), mode='constant')
    
    # 计算输出尺寸
    H_out = (padded_H - kh) // sh + 1
    W_out = (padded_W - kw) // sw + 1
    
    # 初始化输出
    output = np.zeros((N, C, H_out, W_out))
    
    # 滑动窗口取最大值
    for i in range(H_out):
        for j in range(W_out):
            h_start = i * sh
            h_end = h_start + kh
            w_start = j * sw
            w_end = w_start + kw
            window = padded_input[:, :, h_start:h_end, w_start:w_end]
            output[:, :, i, j] = np.max(window, axis=(2,3))
    
    # 如果输入原是3维，恢复为3维输出
    if output.shape[0] == 1:
        output = output[0]
    return output

In [2]:
#3.2 编程题
import torch
import torch.nn as nn

class NiNBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, stride, padding):
        super(NiNBlock, self).__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding),
            nn.ReLU(),
            nn.Conv2d(out_channels, out_channels, kernel_size=1),
            nn.ReLU(),
            nn.Conv2d(out_channels, out_channels, kernel_size=1),
            nn.ReLU()
        )
    
    def forward(self, x):
        return self.net(x)

In [3]:
#4.2 编程题
import torch
import torch.nn as nn

class Residual(nn.Module):
    def __init__(self, in_channels, out_channels, use_1x1conv=False, stride=1):
        super(Residual, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1)
        self.bn2 = nn.BatchNorm2d(out_channels)
        
        self.use_1x1conv = use_1x1conv
        if use_1x1conv:
            self.conv3 = nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride)
        else:
            # 当输入输出通道数不同或 stride != 1 时，即使 use_1x1conv=False 也需要调整？这里按题目要求仅由标志决定
            self.conv3 = None
    
    def forward(self, x):
        y = nn.functional.relu(self.bn1(self.conv1(x)))
        y = self.bn2(self.conv2(y))
        if self.use_1x1conv:
            x = self.conv3(x)
        y += x
        return nn.functional.relu(y)

In [4]:
#5.2 编程题
from torchvision import transforms

augmentation_pipeline = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.08, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.5, contrast=0.5, saturation=0.5),
    transforms.ToTensor()
])

In [5]:
#6.2 编程题
import torch
import torch.nn.functional as F

def label_smoothing_cross_entropy(logits, labels, epsilon=0.1, num_classes=None):
    """
    logits: (N, C) 未归一化的预测分数
    labels: (N,) 真实类别索引
    epsilon: 平滑因子
    num_classes: 类别总数，若为 None 则从 logits 推断
    """
    if num_classes is None:
        num_classes = logits.shape[-1]
    
    # 创建平滑后的标签分布
    smooth_targets = torch.full_like(logits, epsilon / (num_classes - 1))
    smooth_targets.scatter_(1, labels.unsqueeze(1), 1.0 - epsilon)
    
    # 计算 log_softmax
    log_probs = F.log_softmax(logits, dim=-1)
    # 交叉熵 = - sum(target * log_prob)
    loss = - (smooth_targets * log_probs).sum(dim=-1).mean()
    return loss